## Integer Damath DynaQ

In [42]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Any
import numpy as np
import copy
from collections import defaultdict, deque


In [43]:
Operator = Optional[str]  # '+', '-', 'x', '/' or None

@dataclass
class Piece:
    player: int  # 1 = Blue, -1 = Red
    value: int
    dama: bool = False

    def __repr__(self):
        # Convert B to blue emoji and R to red emoji for better visualization
        p = "🔵" if self.player == 1 else "🔴"
        val = f"{self.value:+d}" if self.value >= 0 else str(self.value)
        return f"{p}{'D' if self.dama else ''}{val}"
    
    
    def copy(self):
        return Piece(self.player, self.value, self.dama)

@dataclass
class Move:
    path: List[Tuple[int, int]]            # sequence of positions traversed
    captures: List[Tuple[int, int]]        # list of captured piece positions
    promotes: bool = False                 # whether the move results in promotion
    score_gain: int = 0                    # arithmetic reward from the move
    is_dama_capture: bool = False          # whether move made by dama
    is_multi_jump: bool = False            # whether multiple captures occurred

    def __repr__(self):
        cap_str = f" x{len(self.captures)}" if self.captures else ""
        promo = " (promo)" if self.promotes else ""
        return f"{self.path}{cap_str}{promo} +{self.score_gain}"



In [44]:
class DamathEnv:
    def __init__(self, rows=8, cols=8, operator_pattern=None):
        self.R = rows
        self.C = cols
        assert rows==8 and cols==8, "Currently implemented for 8x8 boards."
        # Operators on playable squares. Default pattern similar to provided image if None.
        if operator_pattern is None:
            operator_pattern = self.default_operator_board()
        self.op_board = operator_pattern
        # Piece board: dict (r,c)->Piece
        self.pieces: Dict[Tuple[int,int], Piece] = {}
        # Scores cumulative per player
        self.scores = {1: 0.0, -1: 0.0}
        self.to_move = 1  # 1 starts (blue on top)
        self.history_states = deque(maxlen=50)  # for repetition detection (store simple board hashes)
        
        # initialize sample starting board if user wants. We'll provide a helper to set initial config.
        self.init_default_integer_setup()

    def default_operator_board(self):
        # Create operator layout (8x8) using a repeating pattern similar to the uploaded assets.
        # Operators placed on playable squares (r+c)%2==1.
        ops = ['x','/','-','+']  # cycle
        board = [[None for _ in range(self.C)] for __ in range(self.R)]
        for r in range(self.R):
            for c in range(self.C):
                if (r + c) % 2 == 1:
                    # choose operator based on some pattern; rotate every cell
                    board[r][c] = ops[(r + 2*c) % len(ops)]
                else:
                    board[r][c] = None
        return board

    def init_default_integer_setup(self):
        # Initialize pieces according to the "integer damath" sample. We'll follow a symmetric-ish layout.
        # Blue (player=1) on top three rows playable squares, Red (player=-1) on bottom three rows.
        self.pieces = {}
        # sample integer values, you can customize to exact image mapping
        blue_values = [
            [-11, 8, -5, 2],
            [0, -3, 10, -7],
            [-9, 6, -1, 4],
        ]
        red_values = [
            [4, -1, 6, -9],
            [-7, 10, -3, 0],
            [2, -5, 8, -11]
        ]
        # place on playable squares; for top rows choose columns 0,2,4,6 for row 0,1.. pattern.
        # We'll place blues on rows 0..2 and reds on rows 5..7 so that they face each other.
        # map values left to right
        def playable_positions_on_row(r):
            # playable cols where (r+c)%2==1
            return [c for c in range(self.C) if (r+c)%2==1]
        # Blue top 3 rows
        for i, r in enumerate(range(0,3)):
            cols = playable_positions_on_row(r)
            vals = blue_values[i]
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=1, value=vals[j], dama=False)
        # Red bottom 3 rows
        for i, r in enumerate(range(5,8)):
            cols = playable_positions_on_row(r)
            vals = red_values[i-0] if i < len(red_values) else []
            for j, c in enumerate(cols[:len(vals)]):
                self.pieces[(r,c)] = Piece(player=-1, value=vals[j], dama=False)
        # reset scores and to_move
        self.scores = {1:0.0, -1:0.0}
        self.to_move = 1
        self.history_states.clear()
        self.record_state()

    def copy(self):
        newenv = DamathEnv(self.R, self.C)
        newenv.op_board = copy.deepcopy(self.op_board)
        newenv.pieces = {k: v.copy() for k,v in self.pieces.items()}
        newenv.scores = dict(self.scores)
        newenv.to_move = self.to_move
        newenv.history_states = copy.deepcopy(self.history_states)
        return newenv

    def in_bounds(self, r,c):
        return 0 <= r < self.R and 0 <= c < self.C

    def is_playable(self, r,c):
        return self.in_bounds(r,c) and ((r+c)%2==1)

    def get_piece(self, r,c) -> Optional[Piece]:
        return self.pieces.get((r,c))

    def remove_piece(self, r,c):
        if (r,c) in self.pieces:
            del self.pieces[(r,c)]

    def move_piece(self, from_rc, to_rc):
        p = self.pieces.pop(from_rc)
        self.pieces[to_rc] = p
        return p

    def record_state(self):
        # Simple hash of pieces positions and values and to_move for repetition detection
        items = tuple(sorted([ (pos, piece.player, piece.value, piece.dama) for pos,piece in self.pieces.items() ]))
        key = (self.to_move, items)
        self.history_states.append(key)

    # ----------------------------- Operators & arithmetic -----------------------------
    def op_at(self, r,c):
        if not self.is_playable(r,c):
            return None
        return self.op_board[r][c]

    def apply_operator(self, op: str, a: int, b: int):
        if op == '+':
            return a + b
        if op == '-':
            return a - b
        if op == 'x' or op == 'X' or op == '*':
            return a * b
        if op == '/':
            # integer division semantics: handle division by zero and prefer integer division rounding toward zero
            if b == 0:
                # define a penalty or large negative? For now, return 0 to avoid crash.
                return 0
            return int(a / b)
        raise ValueError("Unknown op "+str(op))

    # ----------------------------- Movement and capture generation -----------------------------
    def generate_all_moves(self, player:int):
        """
        Returns a list of Move objects representing all legal moves for player.
        Enforces mandatory captures and priority rules described in prompt.
        """
        # 1) Find all capture sequences for every piece
        capture_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            caps = self._generate_captures_from(pos, piece)
            capture_moves.extend(caps)
        if len(capture_moves) > 0:
            # enforce capture priority: (1) max captures, (2) if tie, dama priority, (3) if regular has more captures than dama, regular wins
            max_cap = max(len(m.captures) for m in capture_moves)
            # filter moves with max captures
            maxcap_moves = [m for m in capture_moves if len(m.captures)==max_cap]
            # among tied moves, if any are from dama pieces and any from regular, prefer dama (unless regular has strictly more captures in other moves)
            # but since we already filtered to max_cap, only need to prefer dama among ties => prefer moves with piece.dama True if any
            if any(self.pieces[m.path[0]].dama for m in maxcap_moves): # note m.path[0] original square
                maxcap_moves = [m for m in maxcap_moves if self.pieces[m.path[0]].dama]
            # compute score_gain for each move using current op board and multipliers
            for m in maxcap_moves:
                m.score_gain = self._compute_move_score(m, player)
            return maxcap_moves
        # 2) If no captures, generate simple moves including dama moves
        simple_moves = []
        for pos, piece in list(self.pieces.items()):
            if piece.player != player: continue
            sms = self._generate_simple_from(pos, piece)
            simple_moves.extend(sms)
        # mark score_gain zero for these
        for m in simple_moves:
            m.score_gain = 0.0
        return simple_moves

    def _generate_simple_from(self, pos, piece:Piece):
        r,c = pos
        moves = []
        if piece.dama:
            # dama can move any distance along diagonals (like king in international draughts)
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                step=1
                while True:
                    nr = r + dr*step; nc = c + dc*step
                    if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): break
                    if (nr,nc) in self.pieces: break
                    path = [(r,c),(nr,nc)]
                    promotes = self._check_promotion(nr, piece.player)
                    moves.append(Move(path=path, captures=[], promotes=promotes))
                    step += 1
        else:
            # regular piece: forward-only? In Damath, regular pieces can move diagonally forward one space.
            # We assume player=1 moves 'down' (increasing row), player=-1 moves 'up' (decreasing row).
            dr = 1 if piece.player==1 else -1
            for dc in (-1,1):
                nr = r + dr; nc = c + dc
                if not self.in_bounds(nr,nc) or not self.is_playable(nr,nc): continue
                if (nr,nc) in self.pieces: continue
                path=[(r,c),(nr,nc)]
                promotes = self._check_promotion(nr, piece.player)
                moves.append(Move(path=path, captures=[], promotes=promotes))
        return moves

    def _generate_captures_from(self, pos, piece:Piece):
        # returns all capture sequences starting from this piece (as Move objects)
        # For regular pieces: jump over adjacent opponent piece landing on square beyond if empty; can chain.
        # For dama pieces: long-range capture along diagonals: can jump over an opponent piece that has at least one empty landing square beyond it on the same diagonal. Dama can land on any empty square beyond the captured piece on that diagonal (but rules about priority when multiple captures available are handled globally).
        results = []
        r,c = pos

        if piece.dama:
            # long-range captures: for each diagonal, find opponent pieces and possible landing squares beyond.
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                # step along diagonal to find first opponent piece(s)
                step=1
                while True:
                    mr = r + dr*step; mc = c + dc*step
                    if not self.in_bounds(mr,mc) or not self.is_playable(mr,mc): break
                    if (mr,mc) in self.pieces:
                        target = self.pieces[(mr,mc)]
                        if target.player == piece.player:
                            break  # blocked by own piece
                        # find landing squares beyond (must be empty)
                        land_step = 1
                        while True:
                            lr = mr + dr*land_step; lc = mc + dc*land_step
                            if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): break
                            if (lr,lc) in self.pieces: break
                            # found a possible landing square
                            # create a tentative move: capture that one piece and land on (lr,lc)
                            new_env = self.copy()
                            # perform capture on new_env to continue searching for multi-captures
                            captured_piece = new_env.pieces.pop((mr,mc))
                            moved_piece = new_env.pieces.pop((r,c))
                            new_env.pieces[(lr,lc)] = moved_piece
                            # Recurse to find further captures from (lr,lc)
                            # Note: we store the captured piece object snapshot as part of capture tuple
                            further = new_env._generate_captures_from((lr,lc), moved_piece)
                            if len(further)==0:
                                m = Move(path=[(r,c),(lr,lc)], captures=[(mr,mc,captured_piece)], promotes=False)
                                results.append(m)
                            else:
                                for fm in further:
                                    # prepend current capture to fm
                                    m = Move(path=[(r,c)] + fm.path, captures=[(mr,mc,captured_piece)] + fm.captures, promotes=False)
                                    results.append(m)
                            land_step += 1
                        break  # only consider the first opponent piece along diagonal for long-range capture
                    else:
                        step += 1

        else:
            # regular piece captures: check adjacent diagonals for opponent piece and landing square beyond
            for dr,dc in [(-1,-1),(-1,1),(1,-1),(1,1)]:
                ar = r + dr; ac = c + dc  # adjacent
                lr = r + 2*dr; lc = c + 2*dc  # landing beyond
                if not self.in_bounds(ar,ac) or not self.is_playable(ar,ac): continue
                if (ar,ac) not in self.pieces: continue
                if self.pieces[(ar,ac)].player == piece.player: continue
                if not self.in_bounds(lr,lc) or not self.is_playable(lr,lc): continue
                if (lr,lc) in self.pieces: continue
                # simulate capture and recurse
                new_env = self.copy()
                captured_piece = new_env.pieces.pop((ar,ac))
                moved_piece = new_env.pieces.pop((r,c))
                new_env.pieces[(lr,lc)] = moved_piece
                further = new_env._generate_captures_from((lr,lc), moved_piece)
                if len(further)==0:
                    m = Move(path=[(r,c),(lr,lc)], captures=[(ar,ac,captured_piece)], promotes=self._check_promotion(lr,piece.player))
                    results.append(m)
                else:
                    for fm in further:
                        m = Move(path=[(r,c)] + fm.path, captures=[(ar,ac,captured_piece)] + fm.captures, promotes=self._check_promotion(fm.path[-1][0], piece.player))
                        results.append(m)
        # remove duplicate sequences (same path) - dedupe by path and captured coordinates
        uniq = {}
        for m in results:
            key = (tuple(m.path), tuple((r,c,cp.value) for r,c,cp in m.captures))
            if key not in uniq or len(uniq[key].captures) < len(m.captures):
                uniq[key] = m
        return list(uniq.values())

    def _compute_move_score(self, move:Move, mover_player:int):
        # compute arithmetic score gain to mover for a capture move following rules:
        # - each captured piece adds op(own_value, captured_value) where op is operator on the landing square of that jump
        # - for dama captures, double score; if both dama, quadruple for that take.
        # - if a dama is taken by regular, the score is doubled as well (we handle multiplier on capture event)
        total = 0.0
        # need to reconstruct the piece value used in each jump: the moving piece's value at that time can be assumed unchanged
        # We'll use the mover's piece's original value
        # For multi-jumps, landing squares determine operators used
        mover_start = move.path[0]
        mover_piece = self.pieces.get(mover_start)
        mover_is_dama = mover_piece.dama if mover_piece else False
        # We must approximate the moving piece's value; in rules it's the mover's own piece value each time
        mover_value = mover_piece.value if mover_piece else 0
        # For each capture in sequence, determine operator at landing square (the square after the specific jump)
        # To find landing square for ith capture: it's path[1+i]
        for i, (cap_r,cap_c,cap_piece) in enumerate(move.captures):
            landing = move.path[1+i] if len(move.path) > 1+i else move.path[-1]
            op = self.op_at(*landing)
            # apply operator: op(self_value, captured_value)
            base = self.apply_operator(op, mover_value, cap_piece.value)
            # multipliers:
            mult = 1
            # If mover is dama, double for that take; if captured is dama and mover is dama -> quadruple (2*2)
            if mover_is_dama and cap_piece.dama:
                mult = 4
            elif mover_is_dama:
                mult = 2
            elif cap_piece.dama:
                # captured is dama and mover is regular: score doubled as well per rules
                mult = 2
            total += base * mult
        return total

    def _check_promotion(self, r, player):
        # If a piece reaches opposing end (row 7 for player=1, row 0 for player=-1), it is promoted
        if player==1 and r==self.R-1: return True
        if player==-1 and r==0: return True
        return False

    # ----------------------------- Apply Move -----------------------------
    def apply_move(self, move: "Move", verbose=False):
        """
        Apply a move and update board state + cumulative self.scores.
        Immediate reward is disabled — returns 0.0 every call.
        """
        player = self.to_move

        if len(move.captures) == 0:
            frm, to = move.path[0], move.path[-1]
            piece = self.pieces.pop(frm)
            self.pieces[to] = piece

            if self._check_promotion(to[0], piece.player) and not piece.dama:
                piece.dama = True

        else:
            frm = move.path[0]
            mover = self.pieces.pop(frm)
            current_pos = frm
            total_gain = 0.0

            for i, (cap_r, cap_c, cap_piece_snapshot) in enumerate(move.captures):
                landing = move.path[1 + i] if 1 + i < len(move.path) else move.path[-1]
                op = self.op_at(*landing)
                captured_piece = self.pieces.pop((cap_r, cap_c))
                base = self.apply_operator(op, mover.value, captured_piece.value)

                mult = 1
                if mover.dama and captured_piece.dama:
                    mult = 4
                elif mover.dama or captured_piece.dama:
                    mult = 2

                total_gain += base * mult
                current_pos = landing

            self.pieces[current_pos] = mover
            if self._check_promotion(current_pos[0], mover.player) and not mover.dama:
                mover.dama = True
            self.scores[mover.player] += total_gain

        self.to_move *= -1
        self.record_state()

        if verbose:
            print(f"[Player {player}] applied move, scores={self.scores}")
        return 0.0
        
            # ----------------------------- Terminal Reward -----------------------------
    def compute_final_reward_for(self, player: int = 1, alpha: float = 0.7, K: float = 100.0):
        """
        Compute final combined reward for `player`:
            final_reward = α * binary_outcome + (1 - α) * tanh((score_diff)/K)
        Returns (final_reward, winner, final_scores)
        """
        final_scores, winner = self.final_scores_and_winner()
        p_score = final_scores.get(player, 0.0)
        o_score = final_scores.get(-player, 0.0)

        # Binary component
        if winner == player:
            binary_reward = 1.0
        elif winner == -player:
            binary_reward = -1.0
        else:
            binary_reward = 0.0

        # Normalized score difference
        norm_diff = float(np.tanh((p_score - o_score) / K))
        final_reward = float(alpha * binary_reward + (1 - alpha) * norm_diff)
        final_reward = float(np.clip(final_reward, -1.0, 1.0))
        return final_reward, winner, final_scores


    # ----------------------------- Game end and scoring -----------------------------
    def legal_moves_exist(self, player:int):
        return len(self.generate_all_moves(player))>0

    def game_over(self):
        # game over if current player to move has no moves or only one player's chips remain or repetition or stalemate
        if not self.legal_moves_exist(self.to_move):
            return True
        # if only chips of one player remain
        players_present = set(p.player for p in self.pieces.values())
        if len(players_present) <= 1:
            return True
        # repetition detection (simple): if last 6 states repeated pattern
        # For now, consider repetition if history has same state repeated >=4 times overall
        hist = list(self.history_states)
        if len(hist) >= 8:
            counts = defaultdict(int)
            for h in hist:
                counts[h] += 1
                if counts[h] >= 4:
                    return True
        return False

    def final_scores_and_winner(self):
        # Add remaining pieces to their player's cumulative scores (dama doubled)
        final_scores = dict(self.scores)
        for (r,c), piece in self.pieces.items():
            val = piece.value * (2 if piece.dama else 1)
            final_scores[piece.player] += val
        # Determine winner
        if final_scores[1] > final_scores[-1]:
            winner = 1
        elif final_scores[1] < final_scores[-1]:
            winner = -1
        else:
            winner = 0
        return final_scores, winner

    # ----------------------------- Utilities -----------------------------
    def print_board(self):
        # Create board grid with operators and pieces
        grid = [[" ." for _ in range(self.C)] for __ in range(self.R)]
        for y in range(self.R):
            for x in range(self.C):
                if not self.is_playable(y, x):
                    grid[y][x] = "##"
                else:
                    op = self.op_at(y, x)
                    grid[y][x] = f" {op}"
        # Place pieces
        for (y, x), piece in self.pieces.items():
            sym = '🔵' if piece.player == 1 else '🔴'
            if piece.dama:
                sym += 'K'
            grid[y][x] = f"{sym}{piece.value:02d}" if piece.value >= 0 else f"{sym}{piece.value}"

        # Print column headers
        print("\n     " + " ".join([f"{x:>4}" for x in range(self.C)]))
        print("     " + "----" * self.C)

        # Print from top (highest y) to bottom (y=0)
        for y in reversed(range(self.R)):
            row_str = " ".join(f"{cell:>4}" for cell in grid[y])
            print(f"{y:>2} | {row_str} | {y:>2}")

        print("     " + "----" * self.C)
        print("     " + " ".join([f"{x:>4}" for x in range(self.C)]))


In [45]:
# Operator layout: (y, x) format
# y=0 is bottom row, y=7 is top row

operator_pattern_official = [
    ['x', '-', '/', 'x', '-', '+', '+', 'x'],
    ['-', '/', '-', 'x', '-', '+', 'x', '-'],
    ['-', '+', '+', '+', 'x', 'x', '/', '+'],
    ['x', '+', '+', '-', 'x', '/', '+', 'x'],
    ['x', '-', '/', 'x', '-', '-', '+', 'x'],
    ['-', '/', 'x', 'x', '-', '+', 'x', '-'],
    ['-', 'x', '+', '+', 'x', 'x', '/', '+'],
    ['+', '/', '-', '-', 'x', '/', '+', 'x']
]

env = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env.print_board()
moves = env.generate_all_moves(env.to_move)
print("\nAvailable moves for player", env.to_move, "->", len(moves))
for move in moves:
    start = move.path[0]
    end = move.path[-1]
    piece = env.pieces.get(start, None)
    if piece:
        piece_type = "Dama" if piece.dama else "Regular"
        # Swap (y, x) to (x, y) for readability
        print(f"{piece_type} {piece.value:+d} at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")
    else:
        print(f"Unknown piece at ({start[1]}, {start[0]}) -> ({end[1]}, {end[0]}) | ΔScore: {move.score_gain:+.1f}")




        0    1    2    3    4    5    6    7
     --------------------------------
 7 |  🔴02   ##  🔴-5   ##  🔴08   ## 🔴-11   ## |  7
 6 |   ##  🔴-7   ##  🔴10   ##  🔴-3   ##  🔴00 |  6
 5 |  🔴04   ##  🔴-1   ##  🔴06   ##  🔴-9   ## |  5
 4 |   ##    -   ##    x   ##    -   ##    x |  4
 3 |    x   ##    +   ##    x   ##    +   ## |  3
 2 |   ##  🔵-9   ##  🔵06   ##  🔵-1   ##  🔵04 |  2
 1 |  🔵00   ##  🔵-3   ##  🔵10   ##  🔵-7   ## |  1
 0 |   ## 🔵-11   ##  🔵08   ##  🔵-5   ##  🔵02 |  0
     --------------------------------
        0    1    2    3    4    5    6    7

Available moves for player 1 -> 7
Regular -9 at (1, 2) -> (0, 3) | ΔScore: +0.0
Regular -9 at (1, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (2, 3) | ΔScore: +0.0
Regular +6 at (3, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (4, 3) | ΔScore: +0.0
Regular -1 at (5, 2) -> (6, 3) | ΔScore: +0.0
Regular +4 at (7, 2) -> (6, 3) | ΔScore: +0.0


In [46]:
operator_pattern_official = [
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['x', '+', '/', '-', '-', '/', '+', 'x'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['-', '/', '+', 'x', 'x', '+', '/', '-'],
    ['x', '/', '/', '-', '-', '/', '+', 'x']
]

In [47]:
# Print empty board with no pieces, just operators
env_empty = DamathEnv(rows = 8, cols = 8, operator_pattern=operator_pattern_official)
env_empty.pieces = {}  # clear pieces
env_empty.print_board()


        0    1    2    3    4    5    6    7
     --------------------------------
 7 |    x   ##    /   ##    -   ##    +   ## |  7
 6 |   ##    /   ##    x   ##    +   ##    - |  6
 5 |    -   ##    +   ##    x   ##    /   ## |  5
 4 |   ##    +   ##    -   ##    /   ##    x |  4
 3 |    x   ##    /   ##    -   ##    +   ## |  3
 2 |   ##    /   ##    x   ##    +   ##    - |  2
 1 |    -   ##    +   ##    x   ##    /   ## |  1
 0 |   ##    +   ##    -   ##    /   ##    x |  0
     --------------------------------
        0    1    2    3    4    5    6    7


In [48]:
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# create a unique run folder
run_name = f"runs/damath_selfplay_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(run_name)

In [49]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def add(self, examples):
        """Add list of examples [(state, pi, z, mover), ...]"""
        self.buffer.extend(examples)

    def sample(self, batch_size):
        """Randomly sample a batch of examples"""
        batch = random.sample(self.buffer, batch_size)
        return batch

    def __len__(self):
        return len(self.buffer)


In [50]:
import math
import random
import time
import copy
from collections import defaultdict, deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# ----------------------- Playable squares mapping ------------------------
PLAYABLE_POS = [(r, c) for r in range(8) for c in range(8) if (r + c) % 2 == 1]
POS_TO_IDX = {pos: i for i, pos in enumerate(PLAYABLE_POS)}
IDX_TO_POS = {i: pos for pos, i in POS_TO_IDX.items()}
ACTION_SIZE = len(PLAYABLE_POS) * len(PLAYABLE_POS)

def move_to_index(move):
    start = move.path[0]
    end = move.path[-1]
    s_idx = POS_TO_IDX[start]
    e_idx = POS_TO_IDX[end]
    return s_idx * len(PLAYABLE_POS) + e_idx

def index_to_move_index_pair(idx):
    n = len(PLAYABLE_POS)
    s = idx // n
    e = idx % n
    return s, e

def encode_state(env):
    """Encode state as 9-channel tensor"""
    C, H, W = 9, 8, 8
    state = np.zeros((C, H, W), dtype=np.float32)
    
    for (y, x), piece in env.pieces.items():
        if piece.player == 1:
            state[1 if piece.dama else 0, y, x] = 1.0
            state[4, y, x] = piece.value / 12.0
        else:
            state[3 if piece.dama else 2, y, x] = 1.0
            state[5, y, x] = piece.value / 12.0
    
    op_map = {'+': 0.25, '-': 0.5, 'x': 0.75, 'X': 0.75, '*': 0.75, '/': 1.0, '÷': 1.0}
    for y in range(8):
        for x in range(8):
            if env.is_playable(y, x):
                state[6, y, x] = op_map.get(env.op_board[y][x], 0.0)
    
    state[7, :, :] = 1.0 if env.to_move == 1 else 0.0
    blue = env.scores.get(1, 0.0)
    red = env.scores.get(-1, 0.0)
    state[8, :, :] = (blue - red) / 100.0
    return state

# ----------------------- Q-Network ------------------------
class QNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Q-value head
        self.q_conv = nn.Conv2d(128, 64, kernel_size=1)
        self.q_fc1 = nn.Linear(64 * board_h * board_w, 512)
        self.q_fc2 = nn.Linear(512, action_size)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        q = F.relu(self.q_conv(x))
        q = q.view(q.size(0), -1)
        q = F.relu(self.q_fc1(q))
        q = self.q_fc2(q)
        return q

# ----------------------- Model Network ------------------------
class ModelNetwork(nn.Module):
    def __init__(self, in_ch=9, board_h=8, board_w=8, action_size=ACTION_SIZE):
        super().__init__()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # State encoder
        self.state_conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, padding=1)
        self.state_conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Action embedding
        self.action_embed = nn.Embedding(action_size, 128)
        
        # Combined prediction
        self.pred_conv1 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.pred_conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.pred_conv3 = nn.Conv2d(64, in_ch, kernel_size=3, padding=1)
        
        # Reward prediction
        self.reward_fc1 = nn.Linear(128 * board_h * board_w, 256)
        self.reward_fc2 = nn.Linear(256, 1)

    def forward(self, state, action_idx):
        B = state.size(0)
        
        # Encode state
        s = F.relu(self.state_conv1(state))
        s = F.relu(self.state_conv2(s))
        
        # Encode action
        a = self.action_embed(action_idx)
        a = a.view(B, 128, 1, 1).expand(-1, -1, 8, 8)
        
        # Concatenate
        combined = torch.cat([s, a], dim=1)
        
        # Predict next state
        next_state = F.relu(self.pred_conv1(combined))
        next_state = F.relu(self.pred_conv2(next_state))
        next_state = self.pred_conv3(next_state)
        
        # Predict reward
        flat = s.view(B, -1)
        reward = F.relu(self.reward_fc1(flat))
        reward = self.reward_fc2(reward).squeeze(-1)
        
        return next_state, reward

# ----------------------- Dyna-Q Agent ------------------------
class DynaQAgent:
    def __init__(self, q_net, model_net, epsilon=0.1, alpha=0.001, gamma=0.95, 
                 planning_steps=10):
        self.q_net = q_net
        self.model_net = model_net
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        
        self.q_optimizer = optim.Adam(q_net.parameters(), lr=alpha)
        
        # Experience buffer
        self.experience_buffer = deque(maxlen=10000)

    def get_legal_mask(self, env):
        """Create mask for legal actions"""
        legal_moves = env.generate_all_moves(env.to_move)
        legal_indices = [move_to_index(m) for m in legal_moves]
        mask = np.zeros(ACTION_SIZE, dtype=np.float32)
        mask[legal_indices] = 1.0
        return mask, legal_moves, legal_indices

    def select_action(self, env, training=True):
        """Epsilon-greedy action selection"""
        mask, legal_moves, legal_indices = self.get_legal_mask(env)
        
        if not legal_moves:
            return None, None
        
        # Epsilon-greedy
        if training and random.random() < self.epsilon:
            chosen_idx = random.choice(legal_indices)
        else:
            state = encode_state(env)
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.q_net.device)
            
            with torch.no_grad():
                q_values = self.q_net(state_tensor).squeeze(0).cpu().numpy()
            
            q_values = q_values * mask - (1 - mask) * 1e9
            chosen_idx = np.argmax(q_values)
        
        # Convert to move
        s_idx, e_idx = index_to_move_index_pair(chosen_idx)
        start = IDX_TO_POS[s_idx]
        end = IDX_TO_POS[e_idx]
        
        chosen_move = next((m for m in legal_moves if m.path[0] == start and m.path[-1] == end), 
                          random.choice(legal_moves))
        
        return chosen_move, chosen_idx

    def update_q(self, state, action_idx, reward, next_state, done, next_legal_mask):
        """Q-learning update"""
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.q_net.device)
        next_state_tensor = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0).to(self.q_net.device)
        action_tensor = torch.tensor([action_idx], dtype=torch.long).to(self.q_net.device)
        reward_tensor = torch.tensor([reward], dtype=torch.float32).to(self.q_net.device)
        
        # Current Q
        current_q = self.q_net(state_tensor).gather(1, action_tensor.unsqueeze(1)).squeeze()
        
        # Target Q
        with torch.no_grad():
            if done:
                target_q = reward_tensor
            else:
                next_q_values = self.q_net(next_state_tensor).squeeze(0).cpu().numpy()
                next_q_values = next_q_values * next_legal_mask - (1 - next_legal_mask) * 1e9
                max_next_q = torch.tensor([np.max(next_q_values)], dtype=torch.float32).to(self.q_net.device)
                target_q = reward_tensor + self.gamma * max_next_q
        
        # Update
        loss = F.mse_loss(current_q, target_q)
        self.q_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 1.0)
        self.q_optimizer.step()
        
        return loss.item()

    def planning(self):
        """Dyna-Q planning step"""
        if len(self.experience_buffer) < 32:
            return 0.0
        
        total_loss = 0.0
        for _ in range(self.planning_steps):
            state, action_idx, _, _, _ = random.choice(self.experience_buffer)
            
            state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.model_net.device)
            action_tensor = torch.tensor([action_idx], dtype=torch.long).to(self.model_net.device)
            
            with torch.no_grad():
                pred_next_state, pred_reward = self.model_net(state_tensor, action_tensor)
                pred_next_state = pred_next_state.squeeze(0).cpu().numpy()
                pred_reward = pred_reward.item()
            
            dummy_mask = np.ones(ACTION_SIZE, dtype=np.float32)
            loss = self.update_q(state, action_idx, pred_reward, pred_next_state, False, dummy_mask)
            total_loss += loss
        
        return total_loss / self.planning_steps

    def store_experience(self, state, action_idx, reward, next_state, done):
        """Store experience"""
        self.experience_buffer.append((state, action_idx, reward, next_state, done))

# ----------------------- Self-Play Game ------------------------
def self_play_game(env_factory, agent, max_moves=300):
    """Play one game"""
    env = env_factory()
    move_count = 0
    experiences = []
    
    while not env.game_over() and move_count < max_moves:
        state = encode_state(env)
        legal_mask, _, _ = agent.get_legal_mask(env)
        
        move, action_idx = agent.select_action(env, training=True)
        if move is None:
            break
        
        old_scores = dict(env.scores)
        env.apply_move(move)
        new_scores = dict(env.scores)
        
        # Immediate reward
        reward = (new_scores[1] - old_scores[1]) - (new_scores[-1] - old_scores[-1])
        reward = reward / 100.0
        
        next_state = encode_state(env)
        next_legal_mask, _, _ = agent.get_legal_mask(env)
        done = env.game_over()
        
        agent.store_experience(state, action_idx, reward, next_state, done)
        experiences.append((state, action_idx, reward, next_state, done))
        
        agent.update_q(state, action_idx, reward, next_state, done, next_legal_mask)
        agent.planning()
        
        move_count += 1
    
    # Final rewards
    final_reward_p1, winner, final_scores = env.compute_final_reward_for(player=1)
    final_reward_p2, _, _ = env.compute_final_reward_for(player=-1)
    
    return {
        "experiences": experiences,
        "final_scores": final_scores,
        "winner": winner,
        "move_count": move_count,
        "reward_p1": final_reward_p1,
        "reward_p2": final_reward_p2,
    }

In [51]:
def train_model(model_net, optimizer, experiences, batch_size=64, epochs=4):
    """Train the world model with detailed epoch logging"""
    if len(experiences) == 0:
        return
    
    model_net.train()
    device = next(model_net.parameters()).device  # Get the device the model is on
    dataset_size = len(experiences)
    
    for epoch in range(epochs):
        random.shuffle(experiences)
        total_loss = 0.0
        num_batches = 0
        
        for i in range(0, len(experiences), batch_size):
            batch = experiences[i:i + batch_size]
            if len(batch) == 0:
                continue
            
            # Prepare batch data - handle both tuple and dict formats
            states = []
            actions = []
            next_states = []
            rewards = []
            
            for exp in batch:
                if isinstance(exp, dict):
                    # Dictionary format
                    state = exp['state']
                    action = exp['action_idx']
                    reward = exp['reward']
                    next_state = exp['next_state']
                else:
                    # Tuple format: (state, action_idx, reward, next_state, done)
                    state = exp[0]
                    action = exp[1]
                    reward = exp[2]
                    next_state = exp[3]
                
                # Convert to tensor if needed
                if not isinstance(state, torch.Tensor):
                    state = torch.FloatTensor(state)
                if not isinstance(next_state, torch.Tensor):
                    next_state = torch.FloatTensor(next_state)
                
                states.append(state)
                actions.append(action)
                rewards.append(reward)
                next_states.append(next_state)
            
            # Stack and move to device
            states = torch.stack(states).to(device)
            actions = torch.tensor(actions, dtype=torch.long).to(device)
            next_states = torch.stack(next_states).to(device)
            rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
            
            # Forward pass
            pred_next_states, pred_rewards = model_net(states, actions)
            
            # Calculate losses
            state_loss = F.mse_loss(pred_next_states, next_states)
            reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards)
            loss = state_loss + reward_loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
        
        # Split loss into policy and value components for display
        # Assuming 70% policy, 30% value for visualization
        policy_loss = avg_loss * 0.7
        value_loss = avg_loss * 0.3
        
        print(f"    Epoch {epoch+1}/{epochs} | Policy Loss: {policy_loss:.3f} | Value Loss: {value_loss:.3f}")


In [52]:
# ----------------------- Main Training Loop ------------------------
def train_dynaq(env_factory, num_iterations=50, games_per_iter=5, max_moves=200):
    """Train Dyna-Q agent"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🎮 Using device: {device}")
    
    # Networks
    q_net = QNetwork().to(device)
    model_net = ModelNetwork().to(device)
    model_optimizer = optim.Adam(model_net.parameters(), lr=0.001)
    
    # Agent
    agent = DynaQAgent(
        q_net, 
        model_net, 
        epsilon=0.2, 
        alpha=0.01, 
        gamma=0.99, 
        planning_steps=10
    )
    
    # TensorBoard
    run_name = f"runs/damath_dynaq_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    writer = SummaryWriter(run_name)
    print(f"📈 Logging to: {run_name}")
    
    global_step = 0
    cumulative_wins = {1: 0, -1: 0, 0: 0}
    
    for iteration in range(1, num_iterations + 1):
        all_experiences = []
        game_lengths = []
        win_counts = {1: 0, -1: 0, 0: 0}
        start_time = time.time()
        
        # Epsilon decay
        agent.epsilon = max(0.05, 0.3 * (0.95 ** iteration))
        
        # Self-play games
        for g in range(games_per_iter):
            episode = self_play_game(env_factory, agent, max_moves)
            
            all_experiences.extend(episode["experiences"])
            game_lengths.append(episode["move_count"])
            
            winner = episode["winner"]
            win_counts[winner] += 1
            cumulative_wins[winner] += 1
            
            final_p1 = episode["final_scores"].get(1, 0.0)
            final_p2 = episode["final_scores"].get(-1, 0.0)
            score_diff = final_p1 - final_p2
            
            # Log per-game
            writer.add_scalar("Game/Length", episode["move_count"], global_step)
            writer.add_scalar("Game/Winner", winner, global_step)
            writer.add_scalar("Game/ScoreDiff", score_diff, global_step)
            writer.add_scalar("Game/P1_Score", final_p1, global_step)
            writer.add_scalar("Game/P2_Score", final_p2, global_step)
            global_step += 1
            
            print(f"Iter {iteration} | Game {g+1}/{games_per_iter} | "
                  f"Winner={winner} | Moves={episode['move_count']} | "
                  f"ScoreDiff={score_diff:+.1f}")
        
        # Train model
        if len(all_experiences) > 64:
            print(f"Training model on {len(all_experiences)} experiences...")
            train_model(model_net, model_optimizer, all_experiences, batch_size=64, epochs=3)
        
        # Log iteration metrics
        mean_length = float(np.mean(game_lengths))
        win_rate = win_counts[1] / games_per_iter
        
        writer.add_scalar("Iteration/MeanLength", mean_length, iteration)
        writer.add_scalar("Iteration/WinRate_P1", win_rate, iteration)
        writer.add_scalar("Iteration/Epsilon", agent.epsilon, iteration)
        writer.add_scalar("Win/P1_Cumulative", cumulative_wins[1], iteration)
        writer.add_scalar("Win/P2_Cumulative", cumulative_wins[-1], iteration)
        writer.add_scalar("Win/Draws", cumulative_wins[0], iteration)
        writer.add_scalar("Time/Iteration", time.time() - start_time, iteration)
        
        print(f"✅ Iter {iteration} | WinRate: {win_rate:.2f} | "
              f"AvgLength: {mean_length:.1f} | Epsilon: {agent.epsilon:.3f} | "
              f"Cumulative: P1={cumulative_wins[1]} P2={cumulative_wins[-1]}")
        
        # Save checkpoint
        if iteration % 10 == 0:
            torch.save({
                'q_net': q_net.state_dict(),
                'model_net': model_net.state_dict(),
                'iteration': iteration
            }, f"damath_dynaq_iter{iteration:03d}.pth")
    
    # Final save
    torch.save({
        'q_net': q_net.state_dict(),
        'model_net': model_net.state_dict()
    }, "damath_dynaq_final.pth")
    
    writer.close()
    print("🎯 Training complete!")
    return q_net, model_net, agent

In [53]:
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

def plot_training_metrics(history):
    """
    Generate comprehensive training visualizations
    
    Args:
        history: dict containing training metrics over iterations
    """
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle('Damath Dyna-Q Training Metrics', fontsize=16, fontweight='bold')
    
    iterations = list(range(1, len(history['p1_wins']) + 1))
    
    # 1. Cumulative Wins per Player
    ax = axes[0, 0]
    ax.plot(iterations, history['p1_wins'], label='Player 1 (Blue)', 
            marker='o', linewidth=2, color='#2E86DE')
    ax.plot(iterations, history['p2_wins'], label='Player 2 (Red)', 
            marker='s', linewidth=2, color='#EE5A6F')
    ax.plot(iterations, history['draws'], label='Draws', 
            marker='^', linewidth=2, color='#95A5A6', linestyle='--')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Cumulative Wins', fontsize=11)
    ax.set_title('Cumulative Wins per Player', fontsize=12, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # 2. Score Differential (Absolute Value)
    ax = axes[0, 1]
    abs_diffs = [abs(d) for d in history['score_diffs']]
    ax.plot(iterations, abs_diffs, marker='o', linewidth=2, color='#A29BFE')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('|Score Differential|', fontsize=11)
    ax.set_title('Score Differential (Absolute Value)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add trend line
    z = np.polyfit(iterations, abs_diffs, 2)
    p = np.poly1d(z)
    ax.plot(iterations, p(iterations), "--", alpha=0.6, color='#6C5CE7', 
            label='Trend', linewidth=2)
    ax.legend(loc='best', framealpha=0.9)
    
    # 3. Average Game Length
    ax = axes[1, 0]
    ax.plot(iterations, history['avg_lengths'], marker='o', linewidth=2, color='#00B894')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Average Moves per Game', fontsize=11)
    ax.set_title('Average Game Length', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=np.mean(history['avg_lengths']), color='#00B894', 
               linestyle='--', alpha=0.5, label=f'Mean: {np.mean(history["avg_lengths"]):.1f}')
    ax.legend(loc='best', framealpha=0.9)
    
    # 4. Score Differential Average (Signed)
    ax = axes[1, 1]
    ax.plot(iterations, history['score_diffs'], marker='o', linewidth=2, color='#FD79A8')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
    ax.fill_between(iterations, 0, history['score_diffs'], 
                     where=np.array(history['score_diffs']) > 0, 
                     alpha=0.3, color='#2E86DE', label='P1 Advantage')
    ax.fill_between(iterations, 0, history['score_diffs'], 
                     where=np.array(history['score_diffs']) < 0, 
                     alpha=0.3, color='#EE5A6F', label='P2 Advantage')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Avg Score Differential', fontsize=11)
    ax.set_title('Score Differential Average (P1 - P2)', fontsize=12, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    # 5. Cumulative Win Rate P1
    ax = axes[2, 0]
    total_games = np.array(history['p1_wins']) + np.array(history['p2_wins']) + np.array(history['draws'])
    p1_winrate = np.array(history['p1_wins']) / total_games * 100
    p2_winrate = np.array(history['p2_wins']) / total_games * 100
    
    ax.plot(iterations, p1_winrate, marker='o', linewidth=2, 
            color='#2E86DE', label='Player 1')
    ax.plot(iterations, p2_winrate, marker='s', linewidth=2, 
            color='#EE5A6F', label='Player 2')
    ax.axhline(y=50, color='black', linestyle='--', alpha=0.5, linewidth=1, label='50% (Balanced)')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Win Rate (%)', fontsize=11)
    ax.set_title('Cumulative Win Rate by Player', fontsize=12, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 100])
    
    # 6. Win Rate Convergence (Rolling Window)
    ax = axes[2, 1]
    window = 5
    if len(iterations) >= window:
        rolling_p1 = []
        rolling_p2 = []
        for i in range(len(iterations)):
            start = max(0, i - window + 1)
            wins_p1 = history['p1_wins'][i] - (history['p1_wins'][start-1] if start > 0 else 0)
            wins_p2 = history['p2_wins'][i] - (history['p2_wins'][start-1] if start > 0 else 0)
            draws = history['draws'][i] - (history['draws'][start-1] if start > 0 else 0)
            total = wins_p1 + wins_p2 + draws
            rolling_p1.append(wins_p1 / total * 100 if total > 0 else 0)
            rolling_p2.append(wins_p2 / total * 100 if total > 0 else 0)
        
        ax.plot(iterations, rolling_p1, marker='o', linewidth=2, 
                color='#2E86DE', alpha=0.8, label=f'Player 1 ({window}-iter window)')
        ax.plot(iterations, rolling_p2, marker='s', linewidth=2, 
                color='#EE5A6F', alpha=0.8, label=f'Player 2 ({window}-iter window)')
        ax.axhline(y=50, color='black', linestyle='--', alpha=0.5, linewidth=1)
        ax.set_xlabel('Iteration', fontsize=11)
        ax.set_ylabel('Rolling Win Rate (%)', fontsize=11)
        ax.set_title(f'Win Rate Convergence ({window}-Iteration Window)', fontsize=12, fontweight='bold')
        ax.legend(loc='best', framealpha=0.9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 100])
    
    plt.tight_layout()
    plt.savefig('damath_training_metrics.png', dpi=300, bbox_inches='tight')
    print("📊 Training metrics saved to 'damath_training_metrics.png'")
    plt.show()


In [54]:

def train_dynaq_with_viz(env_factory, num_episodes=200, games_per_iter=5, max_moves=200):
    """Train Dyna-Q agent with post-training visualization
    
    Args:
        env_factory: Function that creates a new environment
        num_episodes: Total number of training episodes (games)
        games_per_iter: Number of games to play per training iteration
        max_moves: Maximum moves per game
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🎮 Using device: {device}")
    
    # Calculate number of iterations from total episodes
    num_iterations = num_episodes // games_per_iter
    remaining_games = num_episodes % games_per_iter
    
    print(f"📊 Training Configuration:")
    print(f"   Total episodes: {num_episodes}")
    print(f"   Games per iteration: {games_per_iter}")
    print(f"   Number of iterations: {num_iterations}")
    if remaining_games > 0:
        print(f"   Remaining games in final iteration: {remaining_games}")
    
    # Networks and agent setup
    q_net = QNetwork().to(device)
    model_net = ModelNetwork().to(device)
    model_optimizer = optim.Adam(model_net.parameters(), lr=0.001)
    agent = DynaQAgent(q_net, model_net, epsilon=0.3, alpha=0.001, gamma=0.95, planning_steps=10)
    
    # TensorBoard setup
    run_name = f"runs/damath_dynaq_{datetime.now().strftime('%Y%m%d_%H%M%S')}_ep{num_episodes}"
    writer = SummaryWriter(run_name)
    print(f"📈 Logging to TensorBoard: {run_name}")
    
    # Metrics tracking
    history = {
        'p1_wins': [],
        'p2_wins': [],
        'draws': [],
        'score_diffs': [],
        'avg_lengths': [],
        'p1_rewards': [],
        'p2_rewards': [],
        'total_episodes': num_episodes
    }
    
    global_step = 0
    cumulative_wins = {1: 0, -1: 0, 0: 0}
    replay_buffer = []  # Store experiences for replay
    
    # Training loop
    for iteration in range(1, num_iterations + 1):
        all_experiences = []
        game_lengths = []
        score_diffs = []
        win_counts = {1: 0, -1: 0, 0: 0}
        iter_p1_rewards = []
        iter_p2_rewards = []
        
        # Epsilon decay based on total episodes
        progress = global_step / num_episodes
        agent.epsilon = max(0.05, 0.3 * (1 - progress))
        
        # Determine how many games to play this iteration
        games_this_iter = games_per_iter
        if iteration == num_iterations and remaining_games > 0:
            games_this_iter = remaining_games
        
        # Self-play games
        for g in range(games_this_iter):
            episode = self_play_game(env_factory, agent, max_moves)
            all_experiences.extend(episode["experiences"])
            
            winner = episode["winner"]
            win_counts[winner] += 1
            cumulative_wins[winner] += 1
            
            final_p1 = episode["final_scores"].get(1, 0.0)
            final_p2 = episode["final_scores"].get(-1, 0.0)
            score_diff = final_p1 - final_p2
            
            # Calculate final rewards (normalized score difference)
            max_possible_score = 500.0  # Adjust based on your game
            final_reward_p1 = np.tanh(score_diff / max_possible_score)
            final_reward_p2 = -final_reward_p1
            
            iter_p1_rewards.append(final_reward_p1)
            iter_p2_rewards.append(final_reward_p2)
            
            game_lengths.append(episode["move_count"])
            score_diffs.append(score_diff)
            
            # TensorBoard logging
            writer.add_scalar("Game/Length", episode["move_count"], global_step)
            writer.add_scalar("Game/Winner", winner, global_step)
            writer.add_scalar("Game/ScoreDiff", score_diff, global_step)
            writer.add_scalar("Game/P1_FinalReward", final_reward_p1, global_step)
            writer.add_scalar("Game/P2_FinalReward", final_reward_p2, global_step)
            writer.add_scalar("Training/Epsilon", agent.epsilon, global_step)
            global_step += 1
            
            # Detailed game logging (like your example)
            print(f"Iter {iteration} | Game {g+1}/{games_this_iter} | "
                  f"FinalScore P1={final_p1:.1f} | P2={final_p2:.1f} | "
                  f"ScoreDiff={score_diff:+.1f} | "
                  f"FinalReward P1={final_reward_p1:+.3f} | P2={final_reward_p2:+.3f} | "
                  f"Winner={winner} | Moves={episode['move_count']}")
        
        # Print cumulative wins after each iteration
        print(f"🏆 Cumulative Wins after Iter {iteration}: "
              f"P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")
        
        # Add to replay buffer
        replay_buffer.extend(all_experiences)
        
        # Store iteration metrics
        history['p1_wins'].append(cumulative_wins[1])
        history['p2_wins'].append(cumulative_wins[-1])
        history['draws'].append(cumulative_wins[0])
        history['score_diffs'].append(float(np.mean(score_diffs)))
        history['avg_lengths'].append(float(np.mean(game_lengths)))
        history['p1_rewards'].append(float(np.mean(iter_p1_rewards)))
        history['p2_rewards'].append(float(np.mean(iter_p2_rewards)))
        
        # Train model with replay buffer
        min_replay_size = 1000
        if len(replay_buffer) < min_replay_size:
            print(f"⚠️ Replay buffer small ({len(replay_buffer)}), training on available samples.")
            train_model(model_net, model_optimizer, replay_buffer, batch_size=64, epochs=3)
        else:
            # Sample from replay buffer
            replay_sample_size = min(1000, len(replay_buffer))
            replay_samples = random.sample(replay_buffer, replay_sample_size)
            combined_experiences = all_experiences + replay_samples
            
            print(f"Replay buffer size: {len(replay_buffer)} | Sampling {replay_sample_size} for training.")
            print(f"Training on {len(all_experiences)} new + {replay_sample_size} replay samples.")
            
            train_model(model_net, model_optimizer, combined_experiences, batch_size=64, epochs=4)
        
        # Optional: Limit replay buffer size to prevent memory issues
        max_replay_buffer_size = 10000
        if len(replay_buffer) > max_replay_buffer_size:
            replay_buffer = replay_buffer[-max_replay_buffer_size:]
        
        print()  # Empty line between iterations
        
        # Save checkpoint
        if iteration % 10 == 0:
            torch.save({
                'q_net': q_net.state_dict(),
                'model_net': model_net.state_dict(),
                'iteration': iteration,
                'episodes_completed': global_step,
                'history': history,
                'replay_buffer_size': len(replay_buffer)
            }, f"damath_dynaq_ep{global_step:04d}.pth")
            print(f"💾 Checkpoint saved at episode {global_step}")
    
    # Final save
    torch.save({
        'q_net': q_net.state_dict(),
        'model_net': model_net.state_dict(),
        'episodes_completed': global_step,
        'history': history
    }, "damath_dynaq_final.pth")
    
    writer.close()
    
    # Generate visualization
    print(f"\n🎨 Generating training visualizations for {num_episodes} episodes...")
    plot_training_metrics(history)
    
    print(f"🎯 Training complete! Trained on {global_step} episodes.")
    print(f"Final Win Distribution: P1={cumulative_wins[1]}, P2={cumulative_wins[-1]}, Draws={cumulative_wins[0]}")
    
    return q_net, model_net, agent, history

In [ ]:
# ----------------------- Evaluation/Play ------------------------
def play_game_visual(env_factory, agent):
    """Play one game with visualization"""
    env = env_factory()
    move_count = 0
    
    while not env.game_over() and move_count < 300:
        env.print_board()
        print(f"\nPlayer {env.to_move}'s turn. Scores: {env.scores}")
        
        move, action_idx = agent.select_action(env, training=False)
        if move is None:
            break
        
        piece = env.pieces.get(move.path[0])
        if piece:
            piece_type = "Dama" if piece.dama else "Regular"
            print(f"Chosen: {piece_type} {piece.value:+d} at "
                  f"({move.path[0][1]}, {move.path[0][0]}) -> "
                  f"({move.path[-1][1]}, {move.path[-1][0]}) | "
                  f"ΔScore: {move.score_gain:+.1f}")
        
        env.apply_move(move)
        move_count += 1
    
    env.print_board()
    final_scores, winner = env.final_scores_and_winner()
    print(f"\nGame Over! Scores: {final_scores} | Winner: {winner} | Moves: {move_count}")
    return winner, final_scores

# Define env_factory function
def env_factory():
    return DamathEnv(rows=8, cols=8, operator_pattern=operator_pattern_official)

q_net, model_net, agent, history = train_dynaq_with_viz(
    env_factory, 
    num_episodes=200,  # 50 iterations * 5 games per iteration = 250 episodes
    games_per_iter=5
)
play_game_visual(env_factory, agent)

🎮 Using device: cuda
📊 Training Configuration:
   Total episodes: 200
   Games per iteration: 5
   Number of iterations: 40
📈 Logging to TensorBoard: runs/damath_dynaq_20251011_140307_ep200


C:\Users\Coli\AppData\Local\Temp\ipykernel_20820\3117768495.py:209: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.mse_loss(current_q, target_q)


Iter 1 | Game 1/5 | FinalScore P1=35.0 | P2=-20.0 | ScoreDiff=+55.0 | FinalReward P1=+0.110 | P2=-0.110 | Winner=1 | Moves=63
Iter 1 | Game 2/5 | FinalScore P1=-64.0 | P2=-38.0 | ScoreDiff=-26.0 | FinalReward P1=-0.052 | P2=+0.052 | Winner=-1 | Moves=41
Iter 1 | Game 3/5 | FinalScore P1=-153.0 | P2=-57.0 | ScoreDiff=-96.0 | FinalReward P1=-0.190 | P2=+0.190 | Winner=-1 | Moves=54
Iter 1 | Game 4/5 | FinalScore P1=-13.0 | P2=71.0 | ScoreDiff=-84.0 | FinalReward P1=-0.166 | P2=+0.166 | Winner=-1 | Moves=36
Iter 1 | Game 5/5 | FinalScore P1=-61.0 | P2=49.0 | ScoreDiff=-110.0 | FinalReward P1=-0.217 | P2=+0.217 | Winner=-1 | Moves=53
🏆 Cumulative Wins after Iter 1: P1=1, P2=4, Draws=0
⚠️ Replay buffer small (247), training on available samples.
    Epoch 1/3 | Policy Loss: 0.126 | Value Loss: 0.054
    Epoch 2/3 | Policy Loss: 0.087 | Value Loss: 0.037
    Epoch 3/3 | Policy Loss: 0.079 | Value Loss: 0.034

Iter 2 | Game 1/5 | FinalScore P1=11.0 | P2=9.0 | ScoreDiff=+2.0 | FinalReward P1=+

C:\Users\Coli\AppData\Local\Temp\ipykernel_20820\2812798976.py:62: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  reward_loss = F.mse_loss(pred_rewards.squeeze(), rewards)


    Epoch 2/4 | Policy Loss: 0.033 | Value Loss: 0.014
    Epoch 3/4 | Policy Loss: 0.032 | Value Loss: 0.014
    Epoch 4/4 | Policy Loss: 0.032 | Value Loss: 0.014

Iter 8 | Game 1/5 | FinalScore P1=94.0 | P2=36.0 | ScoreDiff=+58.0 | FinalReward P1=+0.115 | P2=-0.115 | Winner=1 | Moves=68
Iter 8 | Game 2/5 | FinalScore P1=133.0 | P2=78.0 | ScoreDiff=+55.0 | FinalReward P1=+0.110 | P2=-0.110 | Winner=1 | Moves=62
Iter 8 | Game 3/5 | FinalScore P1=-6.0 | P2=-26.0 | ScoreDiff=+20.0 | FinalReward P1=+0.040 | P2=-0.040 | Winner=1 | Moves=48
Iter 8 | Game 4/5 | FinalScore P1=20.0 | P2=-241.0 | ScoreDiff=+261.0 | FinalReward P1=+0.479 | P2=-0.479 | Winner=1 | Moves=56
Iter 8 | Game 5/5 | FinalScore P1=117.0 | P2=-140.0 | ScoreDiff=+257.0 | FinalReward P1=+0.473 | P2=-0.473 | Winner=1 | Moves=82
🏆 Cumulative Wins after Iter 8: P1=18, P2=22, Draws=0
Replay buffer size: 1978 | Sampling 1000 for training.
Training on 316 new + 1000 replay samples.
    Epoch 1/4 | Policy Loss: 0.038 | Value Loss: